# M4 — Reduced-Feature Baselines: eICU Retrain + MIMIC-IV Comparison (Manuscript, Issue 2)

Design (from supervisor email, Issue 2):

1. Retrain LR, RF, XGB on eICU training data using **only the 10 GP formula features**
   (bun, pf_ratio, lactate_max, vent, intubated, platelets_min, map_mean, age_numeric,
   temperature, bilirubin) → LR-10, RF-10, XGB-10.
2. Evaluate LR-10/RF-10/XGB-10 on the eICU i.i.d. test set — does discrimination loss vs the
   full 58-feature baselines come from feature count or model structure?
3. Apply GP (zero-shot, already computed in NB15) and LR-10/RF-10/XGB-10 (retrained on eICU,
   applied directly — not zero-shot in GP's sense, but not re-fit on MIMIC either) to the full
   MIMIC-IV cohort on the same 10 features. Fair cross-database comparison.
4. Platt-recalibrate LR-10/RF-10 using a MIMIC-IV validation subset; compare against raw GP.

Model definitions and Platt-calibration approach reused verbatim from
`NB08_baseline_models.ipynb` (logit-regression Platt fit, not `CalibratedClassifierCV`), just
restricted to the 10-feature subset. MIMIC feature assembly (pf_ratio construction, eICU-median
imputation) reused verbatim from `NB15_mimic_external_validation.ipynb` Cell 4.

**Does not modify or re-execute any NB01–NB15 thesis notebook.**

In [1]:
import json
import pickle
import sys
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT = Path(r"C:\ML PROJECT\sepsis-gp")
sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece, compute_metrics

DATA_PROC = PROJECT / "data" / "processed"
MIMIC = Path("C:/ML PROJECT/DATASETS/mimic-iv-3.1")
RUNS_DIR = PROJECT / "results" / "v2_bce" / "gp_runs"
OUT_DIR = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def _logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1.0 - p))

FORMULA_10 = ["bun", "pf_ratio", "lactate_max", "vent", "intubated",
              "platelets_min", "map_mean", "age_numeric", "temperature", "bilirubin"]
print(f"Formula-10 features: {FORMULA_10}")

Formula-10 features: ['bun', 'pf_ratio', 'lactate_max', 'vent', 'intubated', 'platelets_min', 'map_mean', 'age_numeric', 'temperature', 'bilirubin']


## Step 1 — Retrain LR-10, RF-10, XGB-10 on eICU (10-feature subset)

Model definitions from `NB08_baseline_models.ipynb` Cell 2, unchanged except the feature set.

In [2]:
feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
split = pd.read_csv(DATA_PROC / "split_random.csv")
feat = feat.merge(split[["patientunitstayid", "split"]], on="patientunitstayid")

train_df = feat[feat["split"] == "train"].reset_index(drop=True)
val_df = feat[feat["split"] == "val"].reset_index(drop=True)
test_df = feat[feat["split"] == "test"].reset_index(drop=True)

X_train = train_df[FORMULA_10].values
y_train = train_df["hospital_mortality"].values.astype(int)
X_val = val_df[FORMULA_10].values
y_val = val_df["hospital_mortality"].values.astype(int)
X_test = test_df[FORMULA_10].values
y_test = test_df["hospital_mortality"].values.astype(int)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

def make_lr():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=1.0, class_weight="balanced", solver="lbfgs",
                                   max_iter=1000, random_state=42)),
    ])

def make_rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                   class_weight="balanced", random_state=42, n_jobs=-1)

def make_xgb(spw):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                          random_state=42, eval_metric="logloss")

spw = float((y_train == 0).sum()) / float((y_train == 1).sum())

lr10 = make_lr(); lr10.fit(X_train, y_train)
rf10 = make_rf(); rf10.fit(X_train, y_train)
xgb10 = make_xgb(spw); xgb10.fit(X_train, y_train)
print("LR-10, RF-10, XGB-10 fit complete.")

Train: 7,814  Val: 1,117  Test: 2,233


LR-10, RF-10, XGB-10 fit complete.


## Step 2 — Evaluate on eICU i.i.d. test set, compare vs full-feature baselines and GP

In [3]:
def platt_fit_apply(p_val, y_val, p_test):
    cal = LogisticRegression(max_iter=1000)
    cal.fit(_logit(p_val).reshape(-1, 1), y_val)
    return cal.predict_proba(_logit(p_test).reshape(-1, 1))[:, 1]

eicu_results = []
for name, model in [("LR-10", lr10), ("RF-10", rf10), ("XGB-10", xgb10)]:
    p_val = model.predict_proba(X_val)[:, 1]
    p_test = model.predict_proba(X_test)[:, 1]
    m_raw = compute_metrics(y_test, p_test, label=name)
    p_test_platt = platt_fit_apply(p_val, y_val, p_test)
    m_platt = compute_metrics(y_test, p_test_platt, label=name + "_platt")
    eicu_results.append({"model": name, "auroc": m_raw["auroc"], "ece": m_raw["ece_10bin"],
                          "ece_platt": m_platt["ece_10bin"]})

eicu_10_df = pd.DataFrame(eicu_results)

full_baselines = pd.read_csv(PROJECT / "results" / "shared" / "tables" / "baseline_test_metrics.csv")
print("Full-feature baseline columns:", full_baselines.columns.tolist())
print()
print("10-feature retrained baselines (eICU i.i.d. test):")
print(eicu_10_df.to_string(index=False))

Full-feature baseline columns: ['model', 'auroc', 'auprc', 'brier', 'brier_skill_pct', 'ece_10bin', 'cal_slope', 'cal_intercept', 'sensitivity', 'specificity', 'f1', 'threshold']

10-feature retrained baselines (eICU i.i.d. test):


 model  auroc    ece  ece_platt
 LR-10 0.7334 0.2765     0.0094
 RF-10 0.7525 0.1158     0.0192
XGB-10 0.7335 0.1603     0.0257


In [4]:
print("Full-feature (58-col) baseline reference values:")
print(full_baselines.to_string(index=False))
print()
print("GP canonical (seed=14) eICU i.i.d. test AUROC = 0.738, ECE = 0.013 (from README/NB11)")

Full-feature (58-col) baseline reference values:
    model  auroc  auprc  brier  brier_skill_pct  ece_10bin  cal_slope  cal_intercept  sensitivity  specificity     f1  threshold
       LR 0.7675 0.4636 0.1964            -40.0     0.2595     0.8659        -1.6248       0.6817       0.7247 0.4489     0.5107
       RF 0.7767 0.4737 0.1299              7.5     0.1156     1.3553        -0.5218       0.6472       0.7904 0.4832     0.3565
      XGB 0.7679 0.4512 0.1411             -0.5     0.1156     0.7342        -1.0082       0.5809       0.8125 0.4640     0.4172
 LR_platt 0.7675 0.4636 0.1168             16.8     0.0117     0.9357        -0.1108       0.6817       0.7247 0.4489     0.1710
 RF_platt 0.7767 0.4737 0.1153             17.9     0.0203     0.8808        -0.1576       0.6472       0.7904 0.4832     0.2104
APACHE-IV 0.6847 0.3699 0.1336              3.4     0.0674     0.3386        -1.0547       0.5294       0.7860 0.4063     0.2696

GP canonical (seed=14) eICU i.i.d. test AUROC =

## Step 3 — Apply GP (zero-shot) and LR-10/RF-10/XGB-10 to full MIMIC-IV cohort

In [5]:
# Reassemble MIMIC's 10-feature matrix (NB15 Cell 4 logic, restricted to formula features).
cohort = pd.read_parquet(DATA_PROC / "mimic_cohort.parquet")
lab_feat = pd.read_parquet(DATA_PROC / "mimic_lab_features.parquet")
vital_feat = pd.read_parquet(DATA_PROC / "mimic_vital_features.parquet")

mfeat = cohort.merge(lab_feat, on="stay_id", how="left").merge(vital_feat, on="stay_id", how="left")

mfeat["fio2_frac_filled"] = mfeat["fio2_frac"]
mfeat.loc[mfeat["fio2_frac_filled"].isna() & (mfeat["vent"] == 0), "fio2_frac_filled"] = 0.21
mfeat["pf_ratio"] = mfeat["pao2"] / mfeat["fio2_frac_filled"].replace(0, np.nan)

eicu_train_full = pd.read_parquet(DATA_PROC / "features_curated.parquet").merge(
    split[split["split"] == "train"][["patientunitstayid"]], on="patientunitstayid")

impute_cols = ["bun", "pf_ratio", "lactate_max", "bilirubin", "platelets_min", "temperature"]
for col in impute_cols:
    n_miss = mfeat[col].isna().sum()
    med = eicu_train_full[col].median()
    mfeat[col] = mfeat[col].fillna(med)
    print(f"{col:<15}: {n_miss:,} missing -> filled with eICU train median {med:.3f}")

X_mimic_10 = mfeat[FORMULA_10].copy()
y_mimic = mfeat["hospital_expire_flag"].values
assert X_mimic_10.isna().sum().sum() == 0, "unexpected NaNs remain in MIMIC 10-feature matrix"
print(f"\nMIMIC 10-feature matrix: {X_mimic_10.shape}, mortality {y_mimic.mean():.4f}")

bun            : 64 missing -> filled with eICU train median 30.000
pf_ratio       : 1,181 missing -> filled with eICU train median 194.286
lactate_max    : 993 missing -> filled with eICU train median 2.000
bilirubin      : 1,457 missing -> filled with eICU train median 0.700
platelets_min  : 80 missing -> filled with eICU train median 174.000
temperature    : 54 missing -> filled with eICU train median 36.500

MIMIC 10-feature matrix: (6152, 10), mortality 0.2962


In [6]:
mimic_gp = pd.read_csv(DATA_PROC / "mimic_gp_predictions.csv")
assert (mimic_gp["stay_id"].values == mfeat["stay_id"].values).all(), "stay_id order mismatch"
p_gp_mimic = mimic_gp["gp_prob"].values

mimic_step3 = []
for name, model in [("LR-10", lr10), ("RF-10", rf10), ("XGB-10", xgb10)]:
    p = model.predict_proba(X_mimic_10.values)[:, 1]
    m = compute_metrics(y_mimic, p, label=name)
    mimic_step3.append({"model": name, "auroc": m["auroc"], "ece": m["ece_10bin"]})

gp_m = compute_metrics(y_mimic, p_gp_mimic, label="GP")
mimic_step3.append({"model": "GP (zero-shot)", "auroc": gp_m["auroc"], "ece": gp_m["ece_10bin"]})

mimic_step3_df = pd.DataFrame(mimic_step3)
print("MIMIC-IV, same 10 features, GP zero-shot vs LR-10/RF-10/XGB-10 (eICU-trained, applied directly):")
print(mimic_step3_df.to_string(index=False))

MIMIC-IV, same 10 features, GP zero-shot vs LR-10/RF-10/XGB-10 (eICU-trained, applied directly):
         model  auroc    ece
         LR-10 0.7034 0.1408
         RF-10 0.7125 0.0229
        XGB-10 0.6950 0.0842
GP (zero-shot) 0.7087 0.1245


## Step 4 — Platt-recalibrate LR-10/RF-10 on a MIMIC-IV validation subset, compare vs raw GP

In [7]:
from sklearn.model_selection import train_test_split

idx_all = np.arange(len(y_mimic))
idx_mval, idx_mtest = train_test_split(idx_all, test_size=0.7, stratify=y_mimic, random_state=42)
print(f"MIMIC val subset (for Platt fitting): {len(idx_mval)}  |  MIMIC test subset: {len(idx_mtest)}")

step4_rows = []
for name, model in [("LR-10", lr10), ("RF-10", rf10)]:
    p_all = model.predict_proba(X_mimic_10.values)[:, 1]
    p_mval, y_mval = p_all[idx_mval], y_mimic[idx_mval]
    p_mtest, y_mtest = p_all[idx_mtest], y_mimic[idx_mtest]

    ece_raw_test = compute_ece(y_mtest, p_mtest)
    p_mtest_platt = platt_fit_apply(p_mval, y_mval, p_mtest)
    ece_platt_test = compute_ece(y_mtest, p_mtest_platt)

    step4_rows.append({"model": name, "n_val": len(idx_mval), "n_test": len(idx_mtest),
                        "ece_raw": round(ece_raw_test, 4), "ece_mimic_platt": round(ece_platt_test, 4)})

gp_test_only = compute_ece(y_mimic[idx_mtest], p_gp_mimic[idx_mtest])
step4_rows.append({"model": "GP (raw, zero-shot)", "n_val": None, "n_test": len(idx_mtest),
                    "ece_raw": round(gp_test_only, 4), "ece_mimic_platt": None})

step4_df = pd.DataFrame(step4_rows)
print(step4_df.to_string(index=False))
print("\nGP needs ZERO patient-level MIMIC data to reach this ECE; LR-10_platt/RF-10_platt needed",
      f"{len(idx_mval)} labelled MIMIC patients to fit their correction.")

MIMIC val subset (for Platt fitting): 1845  |  MIMIC test subset: 4307


              model  n_val  n_test  ece_raw  ece_mimic_platt
              LR-10 1845.0    4307   0.1397            0.026
              RF-10 1845.0    4307   0.0224            0.015
GP (raw, zero-shot)    NaN    4307   0.1239              NaN

GP needs ZERO patient-level MIMIC data to reach this ECE; LR-10_platt/RF-10_platt needed 1845 labelled MIMIC patients to fit their correction.


In [8]:
eicu_10_df.to_csv(OUT_DIR / "M4_step2_eicu_10feature_baselines.csv", index=False)
mimic_step3_df.to_csv(OUT_DIR / "M4_step3_mimic_zeroshot_comparison.csv", index=False)
step4_df.to_csv(OUT_DIR / "M4_step4_mimic_platt_comparison.csv", index=False)
print("Saved: M4_step2_eicu_10feature_baselines.csv, M4_step3_mimic_zeroshot_comparison.csv,",
      "M4_step4_mimic_platt_comparison.csv")

Saved: M4_step2_eicu_10feature_baselines.csv, M4_step3_mimic_zeroshot_comparison.csv, M4_step4_mimic_platt_comparison.csv


## Findings

Reported after execution below — see the three saved tables. These jointly answer: (a) is GP's
AUROC deficit vs full baselines about feature count or model structure (Step 2 vs full-feature
reference)? (b) does GP's calibration advantage persist under cross-database shift when baselines
use the identical 10 features (Step 3)? (c) does the interpretable formula transfer better than a
retrained-and-recalibrated baseline, and at what data cost (Step 4)?